## Importing Libraries

In [38]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as WDW
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## Set up and configure Selenium Web_Driver

In [39]:
print("Setting up Webdriver....")
chrome_opt = Options() # Initialize the chrome webdriver
chrome_opt.add_argument("--headless")
chrome_opt.add_argument("--disable-gpu")
chrome_opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36")
print("configuration done!")

Setting up Webdriver....
configuration done!


In [40]:
# Setting up webdriver: Installation
print("Installing Chrome webdriver")
service = Service(ChromeDriverManager ().install())
print("Final setup")
driver = webdriver.Chrome(service=service, options=chrome_opt)

Installing Chrome webdriver
Final setup


## Making a connection to the Web page

In [41]:
URL= "https://www.framesdirect.com/eyeglasses"

In [42]:
print(f"Visiting {URL} page")
driver.get(URL)

# Further Instructions
try:
    print("Waiting for product tiles to load")
    WDW(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'fd-cat')))
    print("Done, Proceed!")
except TimeoutError as e:
    print(f"Expected tag did not load on time: {e}")
    pass

Visiting https://www.framesdirect.com/eyeglasses page
Waiting for product tiles to load
Done, Proceed!


In [43]:
content = driver.page_source
page = BeautifulSoup(content, "html.parser")

## Data Extraction

In [44]:

prod_holder = page.find_all('div', class_="prod-holder")
# Filter to only actual products (exclude ads)
prod_holder = [p for p in prod_holder if p.get('data-element-id', '').startswith('Tiles_Tile')]
print(f"Found {len(prod_holder)} products on the page.")

Found 24 products on the page.


In [48]:
product = []

for prod in prod_holder:
    product_title = prod.find('div', class_="prod-title")
    prod_bot = prod.find('div', class_="prod-bot")

    if product_title:
        # brand_name
        brand_name = product_title.find('div', class_="catalog-name")
        brand = brand_name.text if brand_name else "Unknown"

        # product_name
        product_name = product_title.find('div', class_="product_name")
        name = product_name.text if product_name else "Unknown"

        # prices
      

        if prod_bot:
            # current_price
            current_price_tag = prod_bot.find('div', class_="prod-aslowas")
            current_price = float(current_price_tag.text.replace('$','').replace(',','')
                             if current_price_tag else 0.0
                        )
            # former_price
            former_price_tag = prod_bot.find('div', class_="prod-catalog-retail-price")
            former_price = float(former_price_tag.text.replace('$','').replace(',','')
            if former_price_tag.text and former_price_tag else 0.0
        )
            
        else:
            current_price = former_price = "Unknown"
    else:
        brand = name = current_price = former_price = "Unknown"
    data = {
        "brand": brand,
        "name": name,
        "current_price": current_price,
        "former_price": former_price
    }
    print(data)
    # Append the data to the list
    product.append(data)

{'brand': 'Ray-Ban', 'name': 'RB5154 Clubmaster', 'current_price': 155.4, 'former_price': 222.0}
{'brand': 'Oakley', 'name': 'Airdrop', 'current_price': 158.9, 'former_price': 227.0}
{'brand': 'Ray-Ban', 'name': 'RB7047', 'current_price': 123.2, 'former_price': 176.0}
{'brand': 'Ray-Ban', 'name': 'RB3547V Oval', 'current_price': 147.0, 'former_price': 210.0}
{'brand': 'Oakley', 'name': 'Socket 5.5', 'current_price': 158.9, 'former_price': 227.0}
{'brand': 'Ray-Ban', 'name': 'RB5387', 'current_price': 123.2, 'former_price': 176.0}
{'brand': 'Oakley', 'name': 'Overhead', 'current_price': 120.4, 'former_price': 172.0}
{'brand': 'Oakley', 'name': 'Neoastra', 'current_price': 145.6, 'former_price': 208.0}
{'brand': 'Oakley', 'name': 'Thinboard', 'current_price': 120.4, 'former_price': 172.0}
{'brand': 'Oakley', 'name': 'Metal Plate', 'current_price': 242.2, 'former_price': 346.0}
{'brand': 'Gucci', 'name': 'GG0011O', 'current_price': 391.0, 'former_price': 460.0}
{'brand': 'Versace', 'name'

In [46]:
import csv 
import json

# save to json file
with open('framesdirect_products.json', 'w') as json_file:
    json.dump(product, json_file, indent=4)
print("Data saved to framesdirect_products.json")

# save to csv file
with open('framesdirect_products.csv', 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["brand", "name", "current_price", "former_price"])
    writer.writeheader()
    writer.writerows(product)  
print("Data saved to framesdirect_products.csv")

Data saved to framesdirect_products.json
Data saved to framesdirect_products.csv
